# 13.9 新偏好改善時，哪些原能力也要檢查？


假設只教「遇到應拒絕題，哪種拒絕更徹底」，卻沒有正常請求的完成例子。新模型可能把所有題都拒絕。只教活潑，也可能忽略JSON限制。因為新舊任務使用同一組語言模型權重，更新一項可能影響其他項。

先列訓練資料配方，再另列要測的能力。配方是計畫，測量表是證據，兩者不能互換。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
recipe = {"風格": 30, "格式": 30, "應拒絕題": 20, "正常題完成": 20}
metrics = {key: None for key in ["風格", "格式", "誠實", "安全", "正常題完成", "原文字任務"]}
print("每100對的計畫", recipe, "合計", sum(recipe.values()))
for key, value in metrics.items():
    print(key, "尚未量測" if value is None else value)

配方合計每100對：風格30、格式30、應拒絕20、正常完成20。這是手寫例子，不是通用推薦比例。`metrics`各項用`None`，表示未知；0分才是測過而失敗。執行會印出六項「尚未量測」。

相同對數也可能有不同學習量。例如兩邊各100個有效位置的30對，共6000位置；兩邊各10個位置的30對只有600。有效位置指計分的答案token與約定的EOS，不含問題與補長格。資料混合時應把兩種數量都記錄。

訓練後用相同保留題比較前後，至少核對原任務、正常完成、格式與正在教的偏好。混合資料仍需按情境判斷，不能把20%拒絕資料理解成「每題20%機率拒絕」。未測安全或反附和，就應保留未知。

練習將正常完成的20對移到應拒絕，總數仍100。比例改成40與0，測量仍未知；你能指出風險，但需要真正測正常題才能知道它是否發生。

<details>
<summary>補充：來源、完整設定與既有實測紀錄</summary>

本章實跑只更新加法正誤偏好與附加句偏好，沒有重新測整套風格、安全、多輪或反附和能力，因此那些欄位仍應留為未測。兩支正誤偏好模型的最後加法都維持起點的0／7，不能稱為保住了原本已會的算術；格式支線雖有7／7相對改善，也同樣零題生成正確。教材上的機制能跑通，與這份模型達成用途，是兩種不同的結果。

[13.2提到的自然資料支線](https://birdhackor.github.io/tiny-perceptron-vlm/13.2.html)則使用另一個寬度32、單層模型，先做80步chosen回答SFT，再做80步DPO。十道驗證中六道相對差距改善、七道較佳候選排前；十道最後檢查則八道相對改善、四道較佳排前。兩種計數再次不同，且本支線沒有評自由生成品質。原標籤套在截短片段上的限制仍然存在，不把這些分數當成自然對話或安全能力。[完整公開報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/dpo.json)保存每對原始分數、長度、加法生成與來源切分，操作見[T.7](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.7)。

</details>
